# Visual Evaluation - Pixtral 12B

Runs Pixtral 12B VLM on all 26 eval scenes. Writes xlsx to data/analysis_pixtral_12b/.


In [1]:
import sys, os, sqlite3, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / '.gitignore').exists():
        ROOT = p; break
sys.path.insert(0, str(ROOT / 'backend/src'))
os.environ['PROJECT_ROOT'] = str(ROOT)

# Load .env for API keys
env_path = ROOT / 'backend' / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

ABLATION_DIR = ROOT / 'data' / 'ablation_pixtral_12b'
VIDEO_DIR = ROOT / 'data/videos/eval'
GT_PATH = ROOT / 'data/videos/eval/ground_truth.xlsx'

from service.impl.visual_service_impl import VisualServiceImpl
from service.impl.interval_service_impl import IntervalServiceImpl
from service.impl.events_service_impl import queries_for_condition
from utils.database import setup_database
from utils.vlm_client import VLMClient

print(f'Project root: {{ROOT}}')
print(f'Dataset: data/videos/eval')


Project root: {ROOT}
Dataset: data/videos/eval


In [2]:
# ── Constants ──
GRID_ROWS, GRID_COLS = 2, 4
SAMPLING_RATE = 24
VLM_DELAY = 3.0
MAX_RETRIES = 10
PROVIDER = 'mistral'
MODEL = "pixtral-12b-2409"
LABEL = 'pixtral_12b'

DELTAS = {
    "delta_visual_vehicle_escape": 50,
    "delta_visual_loitering": 150,
    "delta_visual_handoff": 240,
}

ANALYSIS_DIR = ROOT / 'data' / 'analysis_pixtral_12b'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# ── Load GT ──
gt = pd.read_excel(GT_PATH, sheet_name='Ground Truth')
gt_visual = gt[gt['modality'] == 'visual'].dropna(subset=['scene'])
gt_visual['scene'] = gt_visual['scene'].astype(int)

expected_df = pd.read_excel(GT_PATH, sheet_name='Expected Events')

print(f'Visual GT rows: {len(gt_visual)}')
print(f'Expected events: {len(expected_df)}')


Visual GT rows: 62
Expected events: 40


In [4]:
# ── Run VLM pipeline + build intervals ──
db_path = ABLATION_DIR / f'pixtral_12b.db'
if db_path.exists():
    db_path.unlink()
conn, cur = setup_database(db_path)
client = VLMClient(provider=PROVIDER, model=MODEL, temperature=0.0, seed=42)
visual = VisualServiceImpl(max_retries=MAX_RETRIES)

for scene in sorted(expected_df['scene'].unique()):
    aid = f'{LABEL}_s{scene}'
    existing = conn.execute(
        'SELECT COUNT(*) FROM VisualPerInterval WHERE AnalysisID = ?', (aid,)
    ).fetchone()[0]
    if existing > 0:
        print(f'  Scene {scene}: skipped ({existing} VPI)')
        continue
    video = VIDEO_DIR / f'scene{scene}.mp4'
    if not video.exists():
        print(f'  Scene {scene}: video not found, skipping')
        continue
    print(f'  Scene {scene}: running VLM...')
    visual.run_pipeline(
        video_path=str(video), conn=conn, client=client,
        grid_rows=GRID_ROWS, grid_cols=GRID_COLS,
        sampling_rate=SAMPLING_RATE, min_interval=VLM_DELAY,
        analysis_id=aid,
        log=print,
    )

conn.close()
print('VLM + interval building done.')


  Scene 1: running VLM...
Starting video analysis: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/videos/eval/scene1.mp4
Provider: mistral, Model: pixtral-12b-2409, Grid: 2x4
Sampling rate: 1 frame every 24 video frames
--- FRAME 0 ---
--- Frame 0 (mistral) ---
Calling mistral API (attempt 1/10)


  -> New object 1 (person)
  -> New object 2 (object)
  -> New object 3 (object)
Analyzing relations...
Rate limiter: waiting 3.00 s before next VLM call


Calling mistral API (attempt 1/10)


Relations: none()
--- FRAME 24 ---
--- Frame 24 (mistral) ---
Rate limiter: waiting 2.97 s before next VLM call


Calling mistral API (attempt 1/10)


...

Calling mistral API (attempt 1/10)


Failed to parse object JSON: Extra data: line 90 column 1 (char 2220)
Analyzing relations...
Rate limiter: waiting 3.00 s before next VLM call


Calling mistral API (attempt 1/10)


Relations: carrying(1, 4) carrying(1, 14)
  -> Saved relation: Frame=240, EventID=1, Type=carrying, Actor=4
  -> Saved relation: Frame=240, EventID=1, Type=carrying, Actor=1
  -> Saved relation: Frame=240, EventID=2, Type=carrying, Actor=1
  -> Saved relation: Frame=240, EventID=2, Type=carrying, Actor=14
VLM analysis complete. 241 video frames processed.
Starting post-processing (sampling rate: 24)...
Loaded 30 vis-relation rows.
Computing duration intervals with merge logic...
Consolidated 9 unique relation intervals.
Filtered to 9 intervals with valid duration; saving...
Intervals saved successfully.
Interval construction complete.


VLM + interval building done.


In [5]:
# ── Event-level evaluation ──
event_rows = []
conn = sqlite3.connect(str(db_path))

for _, row in expected_df.iterrows():
    aid = f'pixtral_12b_s{row["scene"]}'
    evt = row['event']
    sql_map = queries_for_condition("A", DELTAS, analysis_id=aid)
    sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
    df = pd.read_sql_query(sql, conn)
    det = not df.empty
    result = 'TP' if det else 'FN'

    vis_rels = ''
    if det:
        parts = []
        for _, r in df.iterrows():
            rel = evt
            sf = int(r['StartFrame'])
            ef = int(r['EndFrame'])
            parts.append(f'{rel}({sf}-{ef})')
        vis_rels = ', '.join(parts)
    else:
        all_rels = conn.execute(
            'SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?',
            (aid,)
        ).fetchall()
        if all_rels:
            parts = [f'{r}({sf}-{ef})' for r, sf, ef in all_rels]
            vis_rels = ', '.join(parts)

    event_rows.append({
        'scene': row['scene'], 'event': evt,
        'detected': 'YES' if det else 'NO', 'result': result,
        'relations': vis_rels,
    })
# FP pass: check negative scenes
all_scenes = sorted(expected_df['scene'].unique())
for evt_fp in sorted(event_metrics_events := sorted(expected_df['event'].unique())):
    pos_scenes = set(expected_df[expected_df['event'] == evt_fp]['scene'])
    for neg_scene in all_scenes:
        if neg_scene in pos_scenes:
            continue
        aid = f'pixtral_12b_s{neg_scene}'
        sql_map = queries_for_condition("A", DELTAS, analysis_id=aid)
        sql = sql_map.get(evt_fp, 'SELECT 0 WHERE 1=0')
        try:
            df = pd.read_sql_query(sql, conn)
            if not df.empty:
                rel_str = evt_fp + '(' + str(int(df.iloc[0].get('StartFrame', 0))) + '-' + str(int(df.iloc[0].get('EndFrame', 0))) + ')'
                event_rows.append({'scene': neg_scene, 'event': evt_fp,
                    'detected': 'YES', 'result': 'FP',
                    'relations': rel_str})
        except:
            pass

conn.close()
edf = pd.DataFrame(event_rows)
edf['relations'] = edf['relations'].fillna('')

# Summary metrics
metrics = []
for evt in sorted(edf['event'].unique()):
    sub = edf[edf['event'] == evt]
    tpp = len(sub[sub['result'] == 'TP'])
    fpp = len(sub[sub['result'] == 'FP'])
    fnn = len(sub[sub['result'] == 'FN'])
    support = tpp + fnn
    p = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
    r = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    metrics.append({
        'event': evt, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'TP': tpp, 'FP': fpp, 'FN': fnn, 'support': support
    })
metrics_df = pd.DataFrame(metrics)
print('\n=== Event Summary ===')
print(metrics_df.to_string(index=False))

# Write xlsx
with pd.ExcelWriter(ANALYSIS_DIR / f'visual_event_eval_pixtral_12b.xlsx') as writer:
    metrics_df.to_excel(writer, sheet_name='Summary', index=False)
    for sc in sorted(expected_df['scene'].unique()):
        sc_df = edf[edf['scene'] == sc]
        if not sc_df.empty:
            sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)



=== Event Summary ===
               event  precision  recall    f1  TP  FP  FN  support
            distress      0.333     0.8 0.471   4   8   1        5
               fight      1.000     0.6 0.750   3   0   2        5
     gesture_overlap      0.273     0.6 0.375   3   8   2        5
gunshot_or_explosion      1.000     0.2 0.333   1   0   4        5
             handoff      1.000     1.0 1.000   5   0   0        5
           loitering      0.800     0.8 0.800   4   1   1        5
   vehicle_collision      0.571     0.8 0.667   4   3   1        5
      vehicle_escape      0.667     0.4 0.500   2   1   3        5


In [6]:
# ── Ablation summary ──
tp = len(edf[edf['result'] == 'TP'])
fp = len(edf[edf['result'] == 'FP'])
fn = len(edf[edf['result'] == 'FN'])
support = tp + fn
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

conn2 = sqlite3.connect(str(db_path))
vpi = conn2.execute('SELECT COUNT(*) FROM VisualPerInterval').fetchone()[0]
conn2.close()

result_df = pd.DataFrame([{
    'provider': LABEL,
    'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3),
    'TP': tp, 'FP': fp, 'FN': fn, 'support': support,
    'VPI': vpi,
}])
result_df.to_excel(ANALYSIS_DIR / 'summary.xlsx', index=False)
print(f'Summary: P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={tp} FP={fp} FN={fn} VPI={vpi}')


Summary: P=0.553 R=0.650 F1=0.598 TP=26 FP=21 FN=14 VPI=325


In [ ]:
# ── Relation-level evaluation ──
relation_types = {
    'physical_altercation',
    'running', 'enter_or_exit_vehicle', 'carrying', 
    'suspicious_near_vehicle', 'vehicle_collision', 'gunshot_visible',
    'explosion_visible',
}

conn = sqlite3.connect(str(db_path))

rel_rows = []
for scene in sorted(expected_df['scene'].unique()):
    aid = f'pixtral_12b_s{scene}'
    scene_gts = gt_visual[gt_visual['scene'] == scene]
    if scene_gts.empty:
        continue

    # VLM relations from VisualRelation table
    cur = conn.execute(
        'SELECT DISTINCT RelationType FROM VisualRelation WHERE AnalysisID = ?',
        (aid,)
    )
    vlm_rels = {row[0] for row in cur.fetchall()}

    gt_rels = set(scene_gts['class'].unique())

    for rel in sorted(relation_types):
        in_gt = rel in gt_rels
        in_vlm = rel in vlm_rels
        if in_gt and in_vlm:
            result = 'TP'
        elif in_gt and not in_vlm:
            result = 'FN'
        elif not in_gt and in_vlm:
            result = 'FP'
        else:
            result = 'TN'
        rel_rows.append({
            'scene': scene, 'relation': rel,
            'in_gt': 'YES' if in_gt else 'NO',
            'in_vlm': 'YES' if in_vlm else 'NO',
            'result': result,
        })

conn.close()
rdf = pd.DataFrame(rel_rows)

# Summary per relation type
rel_metrics = []
for rel in sorted(rdf['relation'].unique()):
    sub = rdf[rdf['relation'] == rel]
    tp = len(sub[sub['result'] == 'TP'])
    fn = len(sub[sub['result'] == 'FN'])
    fp = len(sub[sub['result'] == 'FP'])
    support = tp + fn
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    rel_metrics.append({
        'relation': rel, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'TP': tp, 'FP': fp, 'FN': fn, 'support': support
    })

rm_df = pd.DataFrame(rel_metrics)
print('\n=== Relation Summary ===')
print(rm_df.to_string(index=False))

with pd.ExcelWriter(ANALYSIS_DIR / f'visual_relation_eval_pixtral_12b.xlsx') as writer:
    rm_df.to_excel(writer, sheet_name='Summary', index=False)
    for sc in sorted(expected_df['scene'].unique()):
        sc_df = rdf[(rdf['scene'] == sc) & (rdf['result'] != 'TN')]
        if not sc_df.empty:
            sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)

print(f'\nDone. XLSX written to {ANALYSIS_DIR}/')



=== Relation Summary ===
               relation  precision  recall    f1  TP  FP  FN  support
               carrying      0.588   1.000 0.741  10   7   0       10
        distressed_face      0.000   0.000 0.000   0   3   0        0
  enter_or_exit_vehicle      0.235   0.800 0.364   4  13   1        5
      explosion_visible      0.750   1.000 0.857   3   1   0        3
     fearful_expression      0.333   0.800 0.471   4   8   1        5
              gesturing      0.167   1.000 0.286   5  25   0        5
        gunshot_visible      1.000   0.333 0.500   1   0   2        3
   physical_altercation      1.000   0.800 0.889   4   0   1        5
                running      0.375   0.600 0.462   6  10   4       10
suspicious_near_vehicle      0.217   1.000 0.357   5  18   0        5
      vehicle_collision      0.571   0.800 0.667   4   3   1        5

Done. XLSX written to /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_pixtral_12b/
